# FRAMEWORKS DE IA — Laboratorio 1 (Nivel Introductorio)
## Tensores, Autograd y tu Primera Red

**Ingeniería Civil Informática** · Unidad 1 · Semanas 1–2

**Autor:** Benjamín Sánchez — Equipo Dinamita

Este notebook cubre:
- Parte 1: Configuración del entorno
- Parte 2: Tensores en PyTorch
- Parte 3: Tensores en TensorFlow
- Parte 4: Red neuronal mínima (1 neurona) en ambos frameworks
- Desafío final: `f(x) = 2x² − 3x + 1` con autograd vs. derivada analítica
- Respuestas a las preguntas

## Parte 1 — Configuración del Entorno

El entorno `frameworks-ia` (Python 3.11) se creó con conda. La instalación usada fue:

```bash
conda create -n frameworks-ia python=3.11 -y
conda activate frameworks-ia

# PyTorch con soporte GPU (CUDA 12.6) — GTX 1650 SUPER
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# TensorFlow
pip install tensorflow

# Herramientas
pip install numpy matplotlib jupyter ipykernel
```

In [1]:
# Verificar instalaciones
import torch
import tensorflow as tf

print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ninguna')
print('TensorFlow:', tf.__version__)

I0000 00:00:1782873630.607545   11182 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


PyTorch: 2.12.1+cu126
CUDA disponible: True
GPU: NVIDIA GeForce GTX 1650 SUPER
TensorFlow: 2.21.0


## Parte 2 — Tensores en PyTorch

### 2.1 Creación y propiedades de tensores
Un tensor es una matriz N-dimensional con soporte para GPU y diferenciación automática.

In [2]:
import torch
import numpy as np

# --- Crear tensores ---
t_zeros  = torch.zeros(3, 4)                        # tensor de ceros 3x4
t_rand   = torch.rand(2, 3)                         # valores aleatorios uniformes [0,1)
t_manual = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # desde lista Python

print('Shape:',  t_rand.shape)   # torch.Size([2, 3])
print('dtype:',  t_rand.dtype)   # torch.float32
print('device:', t_rand.device)  # cpu
print('t_zeros:\n', t_zeros)

Shape: torch.Size([2, 3])
dtype: torch.float32
device: cpu
t_zeros:
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])


In [3]:
# --- Operaciones elementales ---
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print('a + b       =', a + b)          # tensor([5., 7., 9.])
print('a * b       =', a * b)          # multiplicación elemento a elemento
print('dot(a, b)   =', torch.dot(a, b))  # 1*4 + 2*5 + 3*6 = 32

a + b       = tensor([5., 7., 9.])
a * b       = tensor([ 4., 10., 18.])
dot(a, b)   = tensor(32.)


In [4]:
# --- Interoperabilidad con NumPy ---
np_arr    = np.array([1.0, 2.0, 3.0])
t_from_np = torch.from_numpy(np_arr)   # comparten memoria
np_from_t = t_manual.numpy()           # de vuelta a numpy

print('torch desde numpy:', t_from_np)
print('numpy desde torch:\n', np_from_t)

torch desde numpy: tensor([1., 2., 3.], dtype=torch.float64)
numpy desde torch:
 [[1. 2.]
 [3. 4.]]


### 2.2 Autograd: diferenciación automática
PyTorch calcula gradientes automáticamente. Es la base del entrenamiento por *backpropagation*.

In [5]:
# requires_grad=True indica que queremos calcular gradientes respecto a estas variables
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# Función: y = w*x + b  (una neurona sin activación)
y = w * x + b
print('y =', y.item())  # 7.0

# Calcular gradientes
y.backward()

print('grad x (dy/dx = w):', x.grad)  # tensor(3.)
print('grad w (dy/dw = x):', w.grad)  # tensor(2.)
print('grad b (dy/db = 1):', b.grad)  # tensor(1.)

# IMPORTANTE: limpiar gradientes antes del siguiente paso
x.grad.zero_(); w.grad.zero_(); b.grad.zero_()
print('gradientes limpiados ->', x.grad, w.grad, b.grad)

y = 7.0
grad x (dy/dx = w): tensor(3.)
grad w (dy/dw = x): tensor(2.)
grad b (dy/db = 1): tensor(1.)
gradientes limpiados -> tensor(0.) tensor(0.) tensor(0.)


In [6]:
# Comprobación: y = x**2  =>  dy/dx = 2*x
x2 = torch.tensor(4.0, requires_grad=True)
y2 = x2 ** 2
y2.backward()
print('x =', x2.item(), '| grad =', x2.grad.item(), '| 2*x =', 2*x2.item())

x = 4.0 | grad = 8.0 | 2*x = 8.0


## Parte 3 — Tensores en TensorFlow

TensorFlow usa `tf.Tensor`. La sintaxis es distinta pero los conceptos son los mismos.

> **Nota (Desafío "Haz la función correcta"):** el documento base usa `tf.reduce_dot(a, b)`, que **no existe** en TensorFlow. La función correcta para el producto punto es `tf.tensordot(a, b, axes=1)`.

In [7]:
import tensorflow as tf

# --- Crear tensores ---
t_zeros = tf.zeros((3, 4))
t_rand  = tf.random.uniform((2, 3))
t_const = tf.constant([[1.0, 2.0], [3.0, 4.0]])

print('Shape:', t_rand.shape)   # (2, 3)
print('dtype:', t_rand.dtype)   # float32

# --- Operaciones ---
a = tf.constant([1.0, 2.0, 3.0])
b = tf.constant([4.0, 5.0, 6.0])

print('a + b       =', (a + b).numpy())              # [5. 7. 9.]
print('dot(a, b)   =', tf.tensordot(a, b, axes=1).numpy())  # función CORRECTA -> 32.0

Shape: (2, 3)
dtype: <dtype: 'float32'>
a + b       = [5. 7. 9.]
dot(a, b)   = 32.0


I0000 00:00:1782873651.613236   11182 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1981 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650 SUPER, pci bus id: 0000:07:00.0, compute capability: 7.5


In [8]:
# --- GradientTape: equivalente al autograd de PyTorch ---
x     = tf.Variable(2.0)   # Variable = tensor entrenable
w     = tf.Variable(3.0)
b_val = tf.Variable(1.0)

with tf.GradientTape() as tape:
    y = w * x + b_val

grads = tape.gradient(y, [x, w, b_val])
print('grad x:', grads[0].numpy())   # 3.0
print('grad w:', grads[1].numpy())   # 2.0
print('grad b:', grads[2].numpy())   # 1.0

grad x: 3.0
grad w: 2.0
grad b: 1.0


## Parte 4 — Red Neuronal Mínima (1 neurona)

### 4.1 Una neurona en PyTorch
`nn.Linear` implementa una capa densa: `y = xWᵀ + b`.

In [9]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Una capa lineal: 3 entradas -> 1 salida
neurona = nn.Linear(in_features=3, out_features=1)

print('Pesos:', neurona.weight)   # shape (1, 3)
print('Bias:',  neurona.bias)     # shape (1,)

# Forward pass con un batch de 5 ejemplos
x = torch.rand(5, 3)   # 5 muestras, 3 features
y_pred = neurona(x)
print('Salida shape:', y_pred.shape)   # torch.Size([5, 1])
print(y_pred)

Pesos: Parameter containing:
tensor([[-0.0043,  0.3097, -0.4752]], requires_grad=True)
Bias: Parameter containing:
tensor([-0.4249], requires_grad=True)
Salida shape: torch.Size([5, 1])
tensor([[-0.4627],
        [-0.5881],
        [-0.3126],
        [-0.5810],
        [-0.2567]], grad_fn=<AddmmBackward0>)


### 4.2 Una neurona en Keras
`tf.keras.layers.Dense` es el equivalente de `nn.Linear` en TensorFlow.

In [10]:
import tensorflow as tf
import numpy as np

# Una capa densa: 3 entradas -> 1 salida
neurona = tf.keras.layers.Dense(units=1)

# Forward pass (la capa se construye con la primera llamada)
x = np.random.rand(5, 3).astype(np.float32)
y_pred = neurona(x)
print('Salida shape:', y_pred.shape)   # (5, 1)
print(y_pred.numpy())

print('Pesos (kernel):', neurona.kernel.shape)   # (3, 1)
print('Bias:',           neurona.bias.shape)     # (1,)

W0000 00:00:1782873659.831023   11275 nvptx_libdevice_path.cc:41] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykernel_launcher.runfiles/cuda_nvdisasm
  ipykernel_launcher.runfiles/nvidia_nvshmem
  ipykernel_launcher.runfiles/cuda_nvvm
  ipykernel_launcher.runfiles/cuda_cudart
  /usr/local/cuda
  /opt/cuda
  /home/benjamin-sanchez/miniconda3/envs/frameworks-ia/lib/python3.11/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc
  /home/benjamin-sanchez/miniconda3/envs/frameworks-ia/lib/python3.11/site-packages/tensorflow/python/platform/../../../../nvidia/cuda_nvcc
  /home/benjamin-sanchez/miniconda3/envs/frameworks-ia/lib/python3.11/site-packages/tensorflow/python/platform/../../cuda
  /home/benjamin-sanchez/miniconda3/envs/frameworks-ia/lib/

Salida shape: (5, 1)
[[-0.31273556]
 [-0.04652058]
 [-0.09704208]
 [ 0.04802442]
 [-0.37853044]]
Pesos (kernel): (3, 1)
Bias: (1,)


## Desafío Final / Ejercicio integrador

Implementa `f(x) = 2x² − 3x + 1` como grafo computacional en PyTorch. Calcula `df/dx` en `x = 1.0, 2.0, 3.0` con autograd y verifica contra la derivada analítica `f'(x) = 4x − 3`.

In [11]:
import torch

def f(x):
    return 2 * x**2 - 3 * x + 1

def df_analitica(x):
    return 4 * x - 3

print(f"{'x':>5} | {'f(x)':>8} | {'autograd':>10} | {'analitica':>10} | coincide")
print('-' * 55)
for valor in [1.0, 2.0, 3.0]:
    x = torch.tensor(valor, requires_grad=True)
    y = f(x)
    y.backward()
    grad_auto = x.grad.item()
    grad_ana  = df_analitica(valor)
    coincide  = torch.isclose(torch.tensor(grad_auto), torch.tensor(grad_ana)).item()
    print(f'{valor:>5} | {y.item():>8.2f} | {grad_auto:>10.4f} | {grad_ana:>10.4f} | {coincide}')

    x |     f(x) |   autograd |  analitica | coincide
-------------------------------------------------------
  1.0 |     0.00 |     1.0000 |     1.0000 | True
  2.0 |     3.00 |     5.0000 |     5.0000 | True
  3.0 |    10.00 |     9.0000 |     9.0000 | True


### ¿Por qué coinciden los gradientes con la derivada analítica?

Coinciden **exactamente**. Autograd no aproxima numéricamente la derivada: construye un **grafo computacional** de las operaciones (`x² → ·2 → −3x → +1`) y, al llamar `backward()`, aplica la **regla de la cadena** propagando derivadas *exactas* de cada operación elemental desde la salida hacia `x`.

Para `f(x) = 2x² − 3x + 1`, la derivada exacta de cada término es `d(2x²)/dx = 4x`, `d(−3x)/dx = −3`, `d(1)/dx = 0`, es decir `f'(x) = 4x − 3`. Como autograd compone estas mismas reglas simbólicas por operación (no usa diferencias finitas), el resultado es idéntico a la derivada analítica salvo errores de redondeo de punto flotante (del orden de 1e-7), que es justo lo que verifica `torch.isclose`.

## Preguntas

**1. ¿Qué diferencia hay entre un tensor de PyTorch con `requires_grad=True` y uno sin él?**

Con `requires_grad=True`, PyTorch **rastrea todas las operaciones** que involucran ese tensor y construye el grafo computacional necesario para calcular gradientes; tras un `backward()` el gradiente queda en `.grad`. Sin él (valor por defecto `False`), el tensor es solo datos: no se registra su historial ni se puede derivar respecto a él, lo que ahorra memoria y cómputo (útil en inferencia, con `torch.no_grad()`).

**2. ¿Por qué TensorFlow usa `GradientTape` en lugar de llamar directamente a `.backward()`?**

TensorFlow 2 usa ejecución *eager* y no guarda el historial de operaciones por defecto (a diferencia de PyTorch). `GradientTape` es un contexto explícito que **"graba"** las operaciones realizadas dentro del bloque `with`; luego `tape.gradient(y, [vars])` recorre esa cinta para calcular los gradientes. Es un diseño explícito: solo se rastrea lo que ocurre dentro del tape, lo que da control fino y evita overhead cuando no se necesitan gradientes.

**3. ¿Qué pasa con los gradientes si haces dos llamadas a `.backward()` seguidas sin limpiar?**

PyTorch **acumula** (suma) los gradientes en `.grad` en lugar de reemplazarlos. Una segunda llamada sumaría el nuevo gradiente al anterior, dando resultados incorrectos. Por eso se llama `optimizer.zero_grad()` o `.grad.zero_()` antes de cada paso. Además, por defecto el grafo se libera tras el primer `backward()`, así que repetirlo sobre el mismo grafo lanza un error salvo que se use `retain_graph=True`.

**4. ¿En qué situaciones preferirías usar NumPy en lugar de tensores?**

Cuando **no** se necesita diferenciación automática ni GPU: preprocesamiento y limpieza de datos, cálculo científico/estadístico clásico, manipulación de arrays en CPU, interoperabilidad con librerías del ecosistema (pandas, scikit-learn, OpenCV, matplotlib) o prototipado ligero. Los tensores (PyTorch/TF) se prefieren cuando hace falta autograd, aceleración por GPU o integración en el entrenamiento de redes neuronales.